# pipeline

> Process a set of Euclid observations through the pipeline.

In [ ]:
# | default_exp euclid.pipeline

In [ ]:
# | exporti


import concurrent.futures
from itertools import product

from nicl.euclid.data_access import DataAccess
from nicl.euclid.utilities import default_data_path
from nicl.euclid.skyflat import create_skyflats, group_obs_ids, write_skyflats
from nicl.euclid.xarray import create_all_zarr_refs, read_all_zarr_refs
from nicl.euclid.persistence import correct_persistence
from nicl.euclid.combine import combine
from nicl.euclid.background_stats import measure

In [ ]:
# | hide
# Additional imports for documentation
from nbdev.showdoc import show_doc

In [ ]:
# Default parameters

RELEASE_NAME = "OTF"
ESAC_SERVER_URL = "https://easotf.esac.esa.int"
PROCESSING_VERSION = "v0.5"

SKYFLAT_N_PIX_NIR = 51
SKYFLAT_FILTER_SIZE_NIR = None
SKYFLAT_N_PIX_VIS = 32
SKYFLAT_FILTER_SIZE_VIS = 3
SKYFLAT_HALF_WINDOW_NIR = 3
SKYFLAT_HALF_WINDOW_VIS = 5

MAX_WORKERS = 8
NIR_FILTERS = ["H", "J", "Y"]
FILTERS = NIR_FILTERS + ["VIS"]

In [ ]:
# | export


def possibly_concurrent(fn, targets, *args, executor=None, **kwargs):
    """Run a function for each target, in parallel if executor is provided."""
    futures = []
    if executor is not None:
        for target in targets:
            futures.append(executor.submit(fn, target, *args, **kwargs))

        # Wait for all futures to complete
        results = []
        for future in concurrent.futures.as_completed(futures):
            try:
                results.append(future.result())
            except Exception as exc:
                print(f"Task generated an exception: {exc}")
        return results
    else:
        results = []
        for target in targets:
            try:
                results.append(fn(target, *args, **kwargs))
            except Exception as exc:
                print(f"Task generated an exception: {exc}")
        return results


def get_required_obs_ids(target_obs_ids, available_obs_ids, half_window):
    """Get the list of obs_ids including neighboring observations required for skyflats"""
    group_for_obs_id = group_obs_ids(
        target_obs_ids, available_obs_ids, half_window=half_window
    )
    obs_ids = [obs_id for group in group_for_obs_id.values() for obs_id in group]
    obs_ids += target_obs_ids
    obs_ids = sorted(list(set(obs_ids)))
    return obs_ids


class Pipeline:
    """Process a set of Euclid observations through the pipeline.

     This class provides methods to process Euclid observations through various
     pipeline steps including data download, skyflat creation, persistence correction,
     stack creation, and background statistics calculation.

     The pipeline can be run as a context manager, which will ensure the parallel
     workers are cleaned up when the context is exited.
     ```python
     with Pipeline(target_obs_ids) as p:
         p.run_nir()
    ```
    """

    def __init__(
        self,
        target_obs_ids,
        release_name=RELEASE_NAME,
        esac_server_url=ESAC_SERVER_URL,
        processing_version=PROCESSING_VERSION,
        skyflat_n_pix_nir=SKYFLAT_N_PIX_NIR,
        skyflat_filter_size_nir=SKYFLAT_FILTER_SIZE_NIR,
        skyflat_n_pix_vis=SKYFLAT_N_PIX_VIS,
        skyflat_filter_size_vis=SKYFLAT_FILTER_SIZE_VIS,
        skyflat_half_window_nir=SKYFLAT_HALF_WINDOW_NIR,
        skyflat_half_window_vis=SKYFLAT_HALF_WINDOW_VIS,
        max_workers=MAX_WORKERS,
        filters=FILTERS,
    ):
        self.target_obs_ids = target_obs_ids
        self.release_name = release_name
        self.esac_server_url = esac_server_url
        self.processing_version = processing_version
        self.skyflat_n_pix_nir = skyflat_n_pix_nir
        self.skyflat_filter_size_nir = skyflat_filter_size_nir
        self.skyflat_n_pix_vis = skyflat_n_pix_vis
        self.skyflat_filter_size_vis = skyflat_filter_size_vis
        self.skyflat_half_window_nir = skyflat_half_window_nir
        self.skyflat_half_window_vis = skyflat_half_window_vis
        self.filters = filters
        self.executor = None
        if max_workers > 1:
            self.executor = concurrent.futures.ProcessPoolExecutor(
                max_workers=max_workers
            )

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.executor is not None:
            self.executor.shutdown(wait=True)
            self.executor = None
        return False

    def __del__(self):
        if self.executor is not None:
            self.executor.shutdown(wait=False)
            self.executor = None

    def get_nir_data(self):
        """Step 1a: Download NIR data files."""
        print("\n=== Downloading NIR Data ===")

        outpath = default_data_path(self.release_name)
        print(f"Saving data to {outpath}")

        da = DataAccess(esac_server_url=self.esac_server_url, release_name=None)
        available_obs_ids = da.find_all_observations()

        nir_obs_ids = get_required_obs_ids(
            self.target_obs_ids, available_obs_ids, self.skyflat_half_window_nir
        )
        print(f"Downloading {len(nir_obs_ids)} NIR observations")
        print("Required NIR obs_ids:", nir_obs_ids)
        for obs_id in nir_obs_ids:
            print(f"Downloading NIR files for {obs_id}", flush=True)
            da.download_calibrated_files_for_observation(
                obs_id, instrument="NISP", outpath=outpath
            )
            da.download_sir_files_for_observation(obs_id, outpath=outpath)

    def get_vis_data(self):
        """Step 1b: Download VIS data files."""
        print("\n=== Downloading VIS Data ===")

        outpath = default_data_path(self.release_name)
        print(f"Saving data to {outpath}")

        da = DataAccess(esac_server_url=self.esac_server_url, release_name=None)
        available_obs_ids = da.find_all_observations()

        vis_obs_ids = get_required_obs_ids(
            self.target_obs_ids, available_obs_ids, self.skyflat_half_window_vis
        )
        print(f"Downloading {len(vis_obs_ids)} VIS observations")
        print("Required VIS obs_ids:", vis_obs_ids)
        for obs_id in vis_obs_ids:
            print(f"Downloading VIS files for {obs_id}", flush=True)
            da.download_calibrated_files_for_observation(
                obs_id, instrument="VIS", outpath=outpath
            )

    def _try_create_nir_zarr_refs(self, obs_id, path, zarr_path):
        """Helper function for creating zarr references."""
        print(f"Creating NIR zarr refs for {obs_id}", flush=True)
        create_all_zarr_refs(path, zarr_path, obs_id_glob=f"{obs_id}")

    def create_nir_skyflats(self):
        """Step 2: Create NIR skyflats."""
        print("\n=== Creating NIR Skyflats ===")

        path = default_data_path(self.release_name, "NIR")
        zarr_path = default_data_path("zarr", self.release_name, "NIR")
        outpath = default_data_path(
            f"{self.release_name}_processed_{self.processing_version}", "skyflat", "NIR"
        )

        da = DataAccess(esac_server_url=self.esac_server_url, release_name=None)
        available_obs_ids = da.find_all_observations()

        # Create zarr references for all observations
        obs_ids = get_required_obs_ids(
            self.target_obs_ids, available_obs_ids, self.skyflat_half_window_nir
        )
        possibly_concurrent(
            self._try_create_nir_zarr_refs,
            obs_ids,
            path=path,
            zarr_path=zarr_path,
            executor=self.executor,
        )

        ds, wcs, _ = read_all_zarr_refs(zarr_path, obs_id_glob=f"*")
        obs_ids = sorted(ds.observation_id.values)
        group_for_obs_id = group_obs_ids(
            obs_ids, obs_ids, half_window=self.skyflat_half_window_nir
        )

        for target_obs_id in self.target_obs_ids:
            for obs_id in range(target_obs_id - 1, target_obs_id + 2):
                if len(list(outpath.glob(f"flat-{obs_id}-*.fits"))) > 0:
                    print(f"Skipping {obs_id} because it already exists")
                    continue
                if obs_id not in group_for_obs_id:
                    print(
                        f"Warning: zarr reference for {obs_id} is not available: skipping"
                    )
                    continue
                print("Creating skyflats for obs_id:", obs_id, flush=True)
                print("Using obs_ids:", group_for_obs_id[obs_id], flush=True)
                flats = create_skyflats(
                    obs_id, group_for_obs_id, zarr_path, n_pix=self.skyflat_n_pix_nir
                )
                write_skyflats(obs_id, flats, outpath, wcs)

    def _try_correct_persistence(self, obs_id, path, processed_path):
        """Helper function for persistence correction."""
        print(f"Processing {obs_id}...", flush=True)
        outpath = processed_path / f"persistence/NIR/{obs_id}/"
        skyflat_path = processed_path / f"skyflat/NIR/"
        correct_persistence(obs_id, path, outpath=outpath, skyflat_path=skyflat_path)
        print(f"Completed {obs_id}...", flush=True)

    def do_persistence_correction(self):
        """Step 3: Perform persistence correction."""
        print("\n=== Performing Persistence Correction ===")
        path = default_data_path(self.release_name)
        processed_path = default_data_path(
            f"{self.release_name}_processed_{self.processing_version}"
        )
        print(f"Processing {len(self.target_obs_ids)} observations:")
        print(self.target_obs_ids)
        possibly_concurrent(
            self._try_correct_persistence,
            self.target_obs_ids,
            path=path,
            processed_path=processed_path,
            executor=self.executor,
        )

    def _try_combine(self, obs_id, in_dir, out_dir_parent, bkg_sub=True):
        """Helper function for creating stacks."""
        out_dir = out_dir_parent / f"{obs_id}"
        if out_dir.exists():
            print(f"Skipping {obs_id} because it already exists", flush=True)
        else:
            print(f"Creating stacks for obs_id: {obs_id}", flush=True)
            try:
                combine(
                    in_dir=in_dir,
                    out_dir=out_dir,
                    obs_ids=obs_id,
                    filters=self.filters,
                    bkg_sub=bkg_sub,
                )
            except Exception as e:
                print(f"Error combining {obs_id}: {e}", flush=True)
                raise

    def create_stacks(self, bkg_sub=True):
        """Step 4: Create image stacks."""
        print("\n=== Creating Stacks ===")
        data_path = default_data_path(
            f"{self.release_name}_processed_{self.processing_version}"
        )
        in_dir = data_path / "persistence" / "NIR"
        if bkg_sub:
            out_dir = data_path / "stacked" / "NIR"
        else:
            out_dir = data_path / "stacked_nobkg" / "NIR"
        possibly_concurrent(
            self._try_combine,
            self.target_obs_ids,
            in_dir=in_dir,
            out_dir_parent=out_dir,
            bkg_sub=bkg_sub,
            executor=self.executor,
        )

    def _try_measure(self, obs_id_and_filter, path, overwrite=False):
        """Helper function for background statistics."""
        obs_id, filter = obs_id_and_filter
        filename = f"EUC_NIR_W-STK_{filter}_{obs_id}.fits"
        obs_path = path / f"{obs_id}"
        if (obs_path / "background_stats" / filename).exists() and not overwrite:
            print(f"Skipping {obs_id} {filter} because it already exists", flush=True)
        else:
            print(f"Measuring background stats for {obs_id} {filter}", flush=True)
            try:
                measure(filename, obs_path)
            except Exception as e:
                print(f"Error measuring {filename}: {e}", flush=True)
                raise

    def calculate_background_stats(self, bkg_sub=True, overwrite=False):
        """Step 5: Calculate background statistics."""
        print("\n=== Calculating Background Statistics ===")
        path = default_data_path(
            f"{self.release_name}_processed_{self.processing_version}"
        )
        if bkg_sub:
            path = path / "stacked" / "NIR"
        else:
            path = path / "stacked_nobkg" / "NIR"
        obs_ids_and_filters = list(product(self.target_obs_ids, self.filters))
        possibly_concurrent(
            self._try_measure,
            obs_ids_and_filters,
            path=path,
            overwrite=overwrite,
            executor=self.executor,
        )

    def run_vis(self):
        """Run a default pipeline for the VIS data.

        You could run a custom version of this with a subset of the steps
        or with different parameters.

        One would typically not want to create stacks for every observation,
        but for a set of targets.

        The VIS and NIR pipelines are independent, so you could run one
        while the other is running.
        """
        print("Starting processing pipeline for VIS data...")

        # Step 1: Download data
        if "VIS" in self.filters:
            self.get_vis_data()

        # Step 2: Create VIS skyflats
        self.create_vis_skyflats()

        # Step 3: Create stacks
        self.create_stacks(bkg_sub=False)

        # Step 4: Calculate background statistics
        self.calculate_background_stats(bkg_sub=False)

    def run_nir(self):
        """Run a default pipeline for the NIR data.

        You could run a custom version of this with a subset of the steps
        or with different parameters.

        One would typically not want to create stacks for every observation,
        but for a set of targets.

        The VIS and NIR pipelines are independent, so you could run one
        while the other is running.
        """
        print("Starting processing pipeline for NIR data...")

        # Step 1: Download data
        if any(x in self.filters for x in NIR_FILTERS):
            self.get_nir_data()

        # Ideally this could run in parallel with the NIR processing
        if "VIS" in self.filters:
            self.get_vis_data()

        # Step 2: Create NIR skyflats
        self.create_nir_skyflats()

        # Step 3: Persistence correction
        self.do_persistence_correction()

        # Step 4: Create stacks
        self.create_stacks(bkg_sub=False)

        # Step 5: Calculate background statistics
        self.calculate_background_stats(bkg_sub=False)

        # Step 1b: Download VIS data for future use
        self.get_vis_data()

        print("\nCirrus processing pipeline complete!")

In [ ]:
show_doc(Pipeline.get_nir_data)

---

### Pipeline.get_nir_data

>      Pipeline.get_nir_data ()

*Step 1a: Download NIR data files.*

In [ ]:
show_doc(Pipeline.get_vis_data)

---

### Pipeline.get_vis_data

>      Pipeline.get_vis_data ()

*Step 1b: Download VIS data files.*

In [ ]:
show_doc(Pipeline.create_nir_skyflats)

---

### Pipeline.create_nir_skyflats

>      Pipeline.create_nir_skyflats ()

*Step 2: Create NIR skyflats.*

In [ ]:
show_doc(Pipeline.do_persistence_correction)

---

### Pipeline.do_persistence_correction

>      Pipeline.do_persistence_correction ()

*Step 3: Perform persistence correction.*

In [ ]:
show_doc(Pipeline.create_stacks)

---

### Pipeline.create_stacks

>      Pipeline.create_stacks (bkg_sub=True)

*Step 4: Create image stacks.*

In [ ]:
show_doc(Pipeline.calculate_background_stats)

---

### Pipeline.calculate_background_stats

>      Pipeline.calculate_background_stats (bkg_sub=True, overwrite=False)

*Step 5: Calculate background statistics.*

In [ ]:
show_doc(Pipeline.run_nir)

---

### Pipeline.run_nir

>      Pipeline.run_nir ()

*Run a default pipeline for the NIR data.

You could run a custom version of this with a subset of the steps
or with different parameters.

One would typically not want to create stacks for every observation,
but for a set of targets.

The VIS and NIR pipelines are independent, so you could run one
while the other is running.*

In [ ]:
show_doc(Pipeline.run_vis)

---

### Pipeline.run_vis

>      Pipeline.run_vis ()

*Run a default pipeline for the VIS data.

You could run a custom version of this with a subset of the steps
or with different parameters.

One would typically not want to create stacks for every observation,
but for a set of targets.

The VIS and NIR pipelines are independent, so you could run one
while the other is running.*